# Lab 4 — Building Blocks, Measured

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s4-building-blocks.ipynb)

One network, one training function, one Weights & Biases project. Each part of this
notebook changes **one key** of a configuration, trains, and sends you to the W&B
workspace to compare the runs: activations, normalisation, dropout and weight decay,
optimisers and schedules, losses, an embedding. The data is MNIST with the pixel
positions and the label meanings permuted — the same kind of data that challenge 8,
*1 Minute Permuted MNIST*, hands your agent. The last three parts turn what you
measured into an `agent.py` that trains and predicts inside two 60-second deadlines,
test it with the challenge's own harness, and submit it.

**Time:** about 15 minutes of compute on a Colab CPU, plus your answers.
**Deliverable:** your copy, run end to end, a one-sentence answer under each part's
question, and your agent on the leaderboard of challenge 8.

---

## 0. Setup

Colab already has PyTorch, torchvision and `wandb`; the cell installs what is missing
(the ML-Arena client, and `wandb` outside Colab). Locally: `uv add torch torchvision
wandb mlarena-sdk`.

**Weights & Biases.** The runs go to a W&B project. A free account
(<https://wandb.ai/signup>, one minute) keeps them and gives you the comparison
workspace this lab is built around: put its key in Colab's *Secrets* panel as
`WANDB_API_KEY` (or in your environment) and allow this notebook to read it. Without
a key, `wandb.login()` asks once — *create an account*, *use an existing one*, or
*don't visualise* — and the last choice writes every run to `./wandb/` in offline
mode; `wandb sync wandb/offline-run-*` uploads them later. Current `wandb` versions
have retired the old anonymous mode (`anonymous="allow"` is a no-op), so there is no
third way. Never paste a key into a cell.

**`QUICK`.** The next cell holds one flag. `QUICK = True` shrinks the subsets and
the epochs so the whole notebook runs in a few minutes — use it to check that
everything works, then set it back to `False` for the numbers you report.

In [ ]:
import importlib.util
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
for module, package in (("wandb", "wandb"), ("mlarena", "mlarena-sdk")):
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import wandb

torch.set_num_threads(3)          # the challenge's agent container has 3 cores: measure like it
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})
pd.set_option("display.width", 140)
from importlib.metadata import version
print(f"torch {torch.__version__}, torchvision {torchvision.__version__}, wandb {version('wandb')}, "
      f"threads {torch.get_num_threads()}, colab {IN_COLAB}")

In [ ]:
QUICK = False      # True: small subsets, few epochs, short budgets -- a smoke test of the whole notebook

In [ ]:
def colab_secret(name):
    """The Colab secret `name`, or None when not in Colab, not set, or not granted to this notebook."""
    if not IN_COLAB:
        return None
    from google.colab import userdata
    try:
        return userdata.get(name)
    except Exception:            # SecretNotFoundError / NotebookAccessError: an optional secret is absent
        return None


PROJECT = "ms2a-s4-building-blocks"
if os.environ.get("WANDB_MODE") != "offline":
    key = os.environ.get("WANDB_API_KEY") or colab_secret("WANDB_API_KEY")
    wandb.login(key=key)          # key=None: asks once (create a free account / use one / don't visualise)
print("W&B mode:", os.environ.get("WANDB_MODE", "online"))

---

## 1. The data: permuted MNIST, as the challenge makes it

Challenge 8 takes MNIST and, for every task, draws a random permutation of the 784
pixel positions and a random permutation of the 10 label meanings, then adds mild
noise. The pixel permutation destroys the spatial structure — a convolution has
nothing to convolve, so the right model is a fully connected network on 784 inputs.
The label permutation removes any prior about digits: the class called `3` is not a
three. `make_task` does the same to torchvision's MNIST, so every number you measure
here transfers to the agent.

The ablations of Parts 1–7 train on `N_TRAIN` images of the MNIST training split and
validate on 10,000 others. The 10,000 test images are never touched before Part 8,
where they play the challenge's `X_test`.

In [ ]:
N_TRAIN, N_VAL, EPOCHS = (4_000, 2_000, 2) if QUICK else (20_000, 10_000, 8)

mnist_train = torchvision.datasets.MNIST("data", train=True, download=True)
mnist_test = torchvision.datasets.MNIST("data", train=False, download=True)


def make_task(images_train, labels_train, images_test, labels_test, seed):
    """Permute the pixel positions and the label meanings, add the challenge's mild noise.

    Returns uint8 arrays shaped like the challenge's: X (N, 28, 28), y_train (N, 1), y_test (N,).
    """
    rng = np.random.RandomState(seed)
    label_perm, pixel_perm = rng.permutation(10), rng.permutation(28 * 28)

    def permute(images):
        flat = images.reshape(len(images), -1)[:, pixel_perm].astype(np.float32) / 255.0
        flat += rng.normal(0, 0.015, flat.shape).astype(np.float32)
        flat = flat * rng.uniform(0.96, 1.04, (len(flat), 1)).astype(np.float32)
        flat += rng.uniform(-0.02, 0.02, (len(flat), 1)).astype(np.float32)
        return (np.clip(flat, 0, 1) * 255).astype(np.uint8).reshape(-1, 28, 28)

    return {"X_train": permute(images_train), "y_train": label_perm[labels_train].reshape(-1, 1).astype(np.int64),
            "X_test": permute(images_test), "y_test": label_perm[labels_test].astype(np.int64)}


def to_tensors(X, y=None):
    """uint8 images -> float32 rows in [0, 1]; labels -> int64."""
    Xt = torch.from_numpy(np.asarray(X).reshape(len(X), -1).astype(np.float32) / 255.0)
    return Xt if y is None else (Xt, torch.from_numpy(np.asarray(y).reshape(-1).astype(np.int64)))


# the ablation task: N_TRAIN + N_VAL images of the training split, no test images
rng = np.random.RandomState(0)
idx = rng.permutation(len(mnist_train))[: N_TRAIN + N_VAL]
task = make_task(mnist_train.data.numpy()[idx], mnist_train.targets.numpy()[idx],
                 mnist_train.data.numpy()[:1], mnist_train.targets.numpy()[:1], seed=0)
X_all, y_all = to_tensors(task["X_train"], task["y_train"])
X_tr, y_tr, X_va, y_va = X_all[:N_TRAIN], y_all[:N_TRAIN], X_all[N_TRAIN:], y_all[N_TRAIN:]
MU, SD = X_tr.mean(), X_tr.std()      # one scalar each, from the training rows only; per-pixel statistics blow up rarely-lit pixels
X_tr, X_va = (X_tr - MU) / SD, (X_va - MU) / SD
print(f"train {tuple(X_tr.shape)}  val {tuple(X_va.shape)}  classes {y_tr.unique().tolist()}")

fig, axes = plt.subplots(1, 2, figsize=(5, 2.6))
axes[0].imshow(mnist_train.data[idx[0]], cmap="gray"); axes[0].set_title(f"MNIST: label {int(mnist_train.targets[idx[0]])}")
axes[1].imshow(task["X_train"][0], cmap="gray"); axes[1].set_title(f"permuted: label {int(task['y_train'][0, 0])}")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

---

## 2. One function: `run(config)`

`CONFIG` names every building block of the network and of its training. `run(changes)`
merges your changes into it, builds the model, trains it, and logs to W&B **once per
epoch**: `train/loss`, `train/acc`, `val/loss`, `val/acc`, the learning rate, the
gradient norm, and — through `wandb.watch` — histograms of every parameter and
gradient. It returns a summary and keeps it in `RESULTS`, so `compare()` prints a table
of every run so far; the trained model stays in `MODELS`, with the watch hooks removed
by `unwatch` so it can be used after its run has finished. Every part below is: change
one key, run, compare.

Two details that the rest of the lab relies on. Validation always runs under
`model.eval()` and `torch.no_grad()`: dropout is off and BatchNorm uses its running
statistics. The gradient norm is read with `clip_grad_norm_(..., max_norm=inf)`, which
returns the total norm without clipping anything.

In [ ]:
CONFIG = dict(
    hidden=(256, 256),      # widths of the hidden layers
    activation="relu",      # relu | gelu | sigmoid
    norm="none",            # none | batchnorm | layernorm  (after the linear layer, before the activation)
    dropout=0.0,            # after the activation
    optimizer="adamw",      # sgd | adamw
    lr=1e-3,
    momentum=0.0,           # sgd only
    weight_decay=0.0,       # decoupled in AdamW, L2 in SGD
    schedule="none",        # none | onecycle
    loss="ce",              # ce | nll | softmax_ce (the mistake) ; label_smoothing applies to ce
    label_smoothing=0.0,
    batch_size=128,
    epochs=EPOCHS,
    seed=0,
)
ACTIVATIONS = {"relu": nn.ReLU, "gelu": nn.GELU, "sigmoid": nn.Sigmoid}
RESULTS, MODELS, HISTORY = {}, {}, {}        # summaries, trained models, per-epoch logs -- by run name


def build_model(cfg, n_in=784, n_out=10):
    layers, d = [], n_in
    for h in cfg["hidden"]:
        layers.append(nn.Linear(d, h))
        if cfg["norm"] == "batchnorm":
            layers.append(nn.BatchNorm1d(h))
        elif cfg["norm"] == "layernorm":
            layers.append(nn.LayerNorm(h))
        layers.append(ACTIVATIONS[cfg["activation"]]())
        if cfg["dropout"] > 0:
            layers.append(nn.Dropout(cfg["dropout"]))
        d = h
    layers.append(nn.Linear(d, n_out))
    return nn.Sequential(*layers)


def build_optimizer(model, cfg, total_steps):
    if cfg["optimizer"] == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=cfg["lr"], momentum=cfg["momentum"], weight_decay=cfg["weight_decay"])
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sched = None
    if cfg["schedule"] == "onecycle":
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg["lr"], total_steps=total_steps)
    return opt, sched


def loss_fn(logits, y, cfg):
    if cfg["loss"] == "ce":
        return F.cross_entropy(logits, y, label_smoothing=cfg["label_smoothing"])
    if cfg["loss"] == "nll":
        return F.nll_loss(F.log_softmax(logits, dim=1), y)
    if cfg["loss"] == "softmax_ce":                    # the mistake: probabilities fed to a loss that expects logits
        return F.cross_entropy(F.softmax(logits, dim=1), y)
    raise ValueError(cfg["loss"])


@torch.no_grad()
def evaluate(model, X, y, cfg, batch=2048):
    model.eval()
    loss, correct = 0.0, 0
    for i in range(0, len(X), batch):
        logits = model(X[i:i + batch])
        loss += loss_fn(logits, y[i:i + batch], cfg).item() * len(logits)
        correct += (logits.argmax(1) == y[i:i + batch]).sum().item()
    return loss / len(X), correct / len(X)


@torch.no_grad()
def dead_units(model, X, batch=2048):
    """Share of first-hidden-layer units active (output > 0) on fewer than 1% of the inputs."""
    model.eval()
    first_act = next(i for i, m in enumerate(model) if isinstance(m, tuple(ACTIVATIONS.values())))
    out = model[: first_act + 1](X[:batch])
    return ((out > 0).float().mean(0) < 0.01).float().mean().item()


def run(changes=None, name=None, tags=(), X_tr=None, y_tr=None, X_va=None, y_va=None):
    """Train one configuration, log every epoch to W&B, return and store its summary."""
    cfg = {**CONFIG, **(changes or {})}
    X_tr, y_tr = (X_tr, y_tr) if X_tr is not None else (globals()["X_tr"], globals()["y_tr"])
    X_va, y_va = (X_va, y_va) if X_va is not None else (globals()["X_va"], globals()["y_va"])
    torch.manual_seed(cfg["seed"])
    model = build_model(cfg)
    steps_per_epoch = math.ceil(len(X_tr) / cfg["batch_size"])
    opt, sched = build_optimizer(model, cfg, steps_per_epoch * cfg["epochs"])
    wb = wandb.init(project=PROJECT, name=name, config=cfg, tags=list(tags))
    wandb.watch(model, log="all", log_freq=steps_per_epoch)      # parameter + gradient histograms, once per epoch
    wandb.define_metric("val/acc", summary="max")
    t0, best, history = time.perf_counter(), 0.0, []
    try:
        for epoch in range(1, cfg["epochs"] + 1):
            model.train()
            loss_sum, correct, grad_norms = 0.0, 0, []
            for idx in torch.randperm(len(X_tr)).split(cfg["batch_size"]):
                logits = model(X_tr[idx])
                loss = loss_fn(logits, y_tr[idx], cfg)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                grad_norms.append(nn.utils.clip_grad_norm_(model.parameters(), max_norm=float("inf")).item())
                opt.step()
                if sched is not None:
                    sched.step()
                loss_sum += loss.item() * len(idx)
                correct += (logits.argmax(1) == y_tr[idx]).sum().item()
            val_loss, val_acc = evaluate(model, X_va, y_va, cfg)
            best = max(best, val_acc)
            history.append({"epoch": epoch, "train/loss": loss_sum / len(X_tr), "train/acc": correct / len(X_tr),
                            "val/loss": val_loss, "val/acc": val_acc, "lr": opt.param_groups[0]["lr"],
                            "grad_norm": float(np.mean(grad_norms)), "seconds": time.perf_counter() - t0})
            wandb.log(history[-1])
        summary = {"val_acc": val_acc, "best_val_acc": best, "train_acc": correct / len(X_tr),
                   "train_loss": loss_sum / len(X_tr), "val_loss": val_loss, "gap": correct / len(X_tr) - val_acc,
                   "grad_norm_last": float(np.mean(grad_norms)), "dead_units": dead_units(model, X_va),
                   "params": sum(p.numel() for p in model.parameters()), "seconds": time.perf_counter() - t0}
        wb.summary.update(summary)
    finally:
        wb.unwatch(model)        # remove the watch hooks: the model is used again after the run has finished
        wandb.finish()
    RESULTS[name or wb.name], MODELS[name or wb.name], HISTORY[name or wb.name] = summary, model, history
    return summary


def compare(*names):
    """A table of the runs named (default: every run so far), best val accuracy first."""
    rows = {k: v for k, v in RESULTS.items() if not names or k in names}
    return pd.DataFrame(rows).T.sort_values("val_acc", ascending=False).round(4)

---

## Part 1 — The baseline

Two hidden layers of 256 ReLU units, AdamW at 1e-3, cross-entropy, no normalisation,
no regularisation: the reference every other run is compared with. Run it, open the
run link W&B prints, and find four panels: `train/loss` and `val/loss` (falling,
together), `val/acc`, `grad_norm`, and under *gradients* the histogram of
`0.weight` — the first layer's gradient — epoch by epoch.

In [ ]:
run({}, name="p1-baseline", tags=["part1"])
compare()

**Question 1.** After the last epoch, how far apart are `train/loss` and `val/loss`,
and what does the `grad_norm` curve do over the epochs — grow, shrink, or plateau?

*Your answer:*

---

## Part 2 — Activations

Same network, three activations. Sigmoid squashes to (0, 1) with a derivative of at
most 0.25, so the gradient shrinks at every layer; ReLU passes gradients unchanged
where its input is positive and blocks them where it is not; GELU is a smooth ReLU.
`dead_units` is the share of first-layer units that are active on fewer than 1% of the
validation inputs — a ReLU unit whose input is negative for every image has zero
gradient and never recovers. The second cell multiplies the learning rate by ten and
shows what that does to the dead-unit share. Compare `grad_norm` across the three
runs in the workspace: the axis is worth setting to log scale.

In [ ]:
for act in ("sigmoid", "relu", "gelu"):
    run({"activation": act}, name=f"p2-{act}", tags=["part2"])
compare("p2-sigmoid", "p2-relu", "p2-gelu")[["val_acc", "grad_norm_last", "dead_units", "seconds"]]

In [ ]:
run({"activation": "relu", "lr": 1e-2}, name="p2-relu-lr1e-2", tags=["part2"])
compare("p2-relu", "p2-relu-lr1e-2")[["val_acc", "train_loss", "grad_norm_last", "dead_units"]]

**Question 2.** Which activation has the smallest gradient norm, by what factor, and
what happened to the dead-unit share and the accuracy of ReLU at ten times the
learning rate?

*Your answer:*

---

## Part 3 — Normalisation

Four hidden layers instead of two, and plain SGD at 0.1 without momentum, where the
depth starts to hurt. BatchNorm standardises each unit over the batch and keeps a
running mean and variance for evaluation; LayerNorm standardises each sample over
its units and has no train/eval difference. Compare `val/acc` after the first epoch
and the `grad_norm` of the three runs.

The second cell is the trap. In `train()` mode BatchNorm uses the statistics of the
batch it is given: four images make bad statistics, one image makes none at all. The
`evaluate` function above calls `model.eval()` before every measurement; forget it and
the validation number depends on the batch size.

In [ ]:
deep = {"hidden": (256, 256, 256, 256), "optimizer": "sgd", "lr": 0.1}
for norm in ("none", "batchnorm", "layernorm"):
    run({**deep, "norm": norm}, name=f"p3-{norm}", tags=["part3"])
compare("p3-none", "p3-batchnorm", "p3-layernorm")[["val_acc", "train_loss", "grad_norm_last", "params", "seconds"]]

In [ ]:
bn = MODELS["p3-batchnorm"]
cfg_bn = {**CONFIG, **deep, "norm": "batchnorm"}
_, acc_eval = evaluate(bn, X_va, y_va, cfg_bn)

bn.train()                                   # the mistake: predicting in training mode
with torch.no_grad():
    acc_train_mode = float(np.mean([(bn(X_va[i:i + 4]).argmax(1) == y_va[i:i + 4]).float().mean().item()
                                    for i in range(0, 400, 4)]))
print(f"eval() mode, any batch size : {acc_eval:.4f}")
print(f"train() mode, batches of 4  : {acc_train_mode:.4f}   (batch statistics of 4 images)")
try:
    bn(X_va[:1])
except ValueError as e:
    print("train() mode, batch of 1    :", e)
_ = bn.eval()

**Question 3.** Which normalisation helped most at depth four with plain SGD, and why
does the BatchNorm network lose accuracy when it predicts in `train()` mode?

*Your answer:*

---

## Part 4 — Dropout and weight decay

Back to the baseline. Dropout zeroes a random 30% of each hidden layer's units at
every training step and does nothing at evaluation; weight decay shrinks every weight
at every step (AdamW applies it directly to the weights, not through the gradient).
Both trade training loss for validation loss. The table's `gap` column is
`train_acc − val_acc`; in the workspace, put `train/loss` and `val/loss` of the four
runs on one panel.

In [ ]:
for name, changes in {"none": {}, "dropout0.3": {"dropout": 0.3}, "wd5e-4": {"weight_decay": 5e-4},
                      "both": {"dropout": 0.3, "weight_decay": 5e-4}}.items():
    run(changes, name=f"p4-{name}", tags=["part4"])
compare("p4-none", "p4-dropout0.3", "p4-wd5e-4", "p4-both")[["val_acc", "train_acc", "gap", "train_loss", "val_loss"]]

**Question 4.** Which run has the smallest gap, and is it also the run with the best
validation accuracy? Name one situation where you would accept the larger gap.

*Your answer:*

---

## Part 5 — Optimisation

Four optimisers on the baseline network: SGD, SGD with momentum 0.9, AdamW, and AdamW
under a OneCycle schedule (the learning rate rises to `lr` over the first 30% of the
steps, then anneals to almost zero). Compare `val/acc` and look at the `lr` panel of
the OneCycle run.

The second cell is a **learning-rate range test**: one pass over the data with the
learning rate multiplied by a constant factor at every step, from 1e-5 to 1, recording
the loss. The loss falls, flattens, then explodes; a good constant learning rate sits
about one decade below the explosion, and a good OneCycle peak a little higher. The
curve is logged to W&B as its own run and plotted here.

In [ ]:
for name, changes in {"sgd": {"optimizer": "sgd", "lr": 0.1},
                      "sgd-momentum": {"optimizer": "sgd", "lr": 0.05, "momentum": 0.9},
                      "adamw": {}, "adamw-onecycle": {"schedule": "onecycle", "lr": 3e-3}}.items():
    run(changes, name=f"p5-{name}", tags=["part5"])
compare("p5-sgd", "p5-sgd-momentum", "p5-adamw", "p5-adamw-onecycle")[["val_acc", "train_loss", "grad_norm_last", "seconds"]]

In [ ]:
def lr_range_test(changes, lr_min=1e-5, lr_max=1.0, steps=None, name="p5-lr-range-test"):
    """Loss against an exponentially rising learning rate; stops once the loss is 4x its minimum."""
    cfg = {**CONFIG, **changes, "lr": lr_min}
    steps = steps or math.ceil(len(X_tr) / cfg["batch_size"])
    torch.manual_seed(cfg["seed"])
    model = build_model(cfg)
    opt, _ = build_optimizer(model, cfg, steps)
    factor = (lr_max / lr_min) ** (1 / steps)
    wandb.init(project=PROJECT, name=name, config=cfg, tags=["part5"])
    lrs, losses, smooth, best = [], [], None, float("inf")
    try:
        model.train()
        for step, idx in enumerate(torch.randperm(len(X_tr)).split(cfg["batch_size"])[:steps]):
            loss = loss_fn(model(X_tr[idx]), y_tr[idx], cfg)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            smooth = loss.item() if smooth is None else 0.9 * smooth + 0.1 * loss.item()
            lr = opt.param_groups[0]["lr"]
            lrs.append(lr); losses.append(smooth); best = min(best, smooth)
            wandb.log({"lr_test/lr": lr, "lr_test/loss": smooth}, step=step)
            if smooth > 4 * best:
                break
            for g in opt.param_groups:
                g["lr"] *= factor
    finally:
        wandb.finish()
    return np.array(lrs), np.array(losses)


lrs, losses = lr_range_test({"optimizer": "sgd", "momentum": 0.9})
LR_TEST = {"lr": lrs.tolist(), "loss": losses.tolist()}
suggested = lrs[losses.argmin()] / 10
plt.figure(figsize=(6, 3.2))
plt.semilogx(lrs, losses, color=PALETTE[0], lw=2)
plt.axvline(suggested, color=PALETTE[1], ls="--", label=f"suggested constant LR ≈ {suggested:.3g}")
plt.xlabel("learning rate (log scale)"); plt.ylabel("smoothed training loss"); plt.legend(); plt.show()
print(f"loss minimum at lr = {lrs[losses.argmin()]:.3g}; explodes past lr ≈ {lrs[-1]:.3g}")

**Question 5.** Which optimiser won at equal epochs, and does the range test's suggested
learning rate agree with the SGD-with-momentum run you launched at 0.05?

*Your answer:*

---

## Part 6 — Losses

`nn.CrossEntropyLoss` is `log_softmax` followed by `nll_loss`, in one numerically stable
step, and it expects **logits**. Feeding it probabilities — a softmax before the
loss — is the most common PyTorch mistake: the network still trains a little, but the
loss has a floor and the gradients are tiny. The first cell shows the equivalence and
the floor numerically, then trains the mistake and a run with label smoothing 0.1,
which spreads 10% of each target over the other classes and keeps the logits from
growing without bound.

The second cell leaves classification: a linear fit to a synthetic target where 5% of
the rows are wrong by +20. MSE squares the residuals, so the outliers pull the line;
L1 does not; Huber is MSE near zero and L1 far from it.

In [ ]:
torch.manual_seed(0)
logits = torch.randn(4, 10) * 3
y = torch.tensor([0, 1, 2, 3])
print(f"cross_entropy            : {F.cross_entropy(logits, y):.6f}")
print(f"nll_loss(log_softmax)    : {F.nll_loss(F.log_softmax(logits, 1), y):.6f}")

perfect = torch.full((1, 10), -30.0); perfect[0, 0] = 30.0          # a network that is certain and right
print(f"\ncertain and right, correct loss        : {F.cross_entropy(perfect, y[:1]):.4f}")
print(f"certain and right, softmax before CE   : {F.cross_entropy(F.softmax(perfect, 1), y[:1]):.4f}"
      f"   (floor = log(1 + 9/e) = {math.log(1 + 9 / math.e):.4f})")

run({"loss": "softmax_ce"}, name="p6-softmax-before-ce", tags=["part6"])
run({"label_smoothing": 0.1}, name="p6-label-smoothing-0.1", tags=["part6"])
compare("p1-baseline", "p6-softmax-before-ce", "p6-label-smoothing-0.1")[["val_acc", "train_loss", "val_loss", "grad_norm_last"]]

In [ ]:
torch.manual_seed(0)
x = torch.rand(400, 1) * 4 - 2
target = 3 * x + 1 + 0.3 * torch.randn_like(x)
target[torch.rand(400) < 0.05] += 20.0                    # 5% of the rows are wrong by +20
losses_reg = {"MSE": nn.MSELoss(), "L1": nn.L1Loss(), "Huber (delta=1)": nn.HuberLoss(delta=1.0)}
rows = {}
for name, criterion in losses_reg.items():
    torch.manual_seed(0)
    lin = nn.Linear(1, 1)
    opt = torch.optim.Adam(lin.parameters(), lr=0.05)
    for _ in range(400):
        opt.zero_grad(); criterion(lin(x), target).backward(); opt.step()
    rows[name] = {"slope": lin.weight.item(), "intercept": lin.bias.item()}
print("truth: slope 3, intercept 1")
pd.DataFrame(rows).T.round(3)

**Question 6.** With softmax before the loss, what did the training loss plateau at,
and what is the smallest value it could ever reach with 10 classes? Which regression
loss recovered the slope best?

*Your answer:*

---

## Part 7 — Embedding

An `nn.Embedding(n, d)` is a lookup table: category *k* becomes row *k*, a `d`-vector
learned with the rest of the network. On the electricity-demand table of Lab 2 the
categorical inputs are the weather condition (4 values), the month (12) and the day
of the week (7). Two networks predict the demand from them plus the mean
temperature and the humidity: one with one-hot inputs, one with embeddings of
dimension 2. The parameter counts are similar here because the cardinalities are
tiny; the cell also prints them for a 10,000-value column, which is where one-hot
inputs stop being an option. The second cell plots the learned month and weekday
vectors and logs the figure to W&B.

In [ ]:
import requests

CSV = "module5_exercise_train.csv"
if not os.path.exists(CSV):
    r = requests.get("https://www.raphaelcousin.com/modules/data-science-practice/module5/exercise/" + CSV, timeout=60)
    r.raise_for_status()
    open(CSV, "wb").write(r.content)

df = pd.read_csv(CSV).drop_duplicates()
df = df[df["electricity_demand"] > 0].copy()                           # Lab 2: one impossible negative day
df["date"] = pd.to_datetime(df["date"]); df = df.sort_values("date")
df["weather_condition"] = df["weather_condition"].ffill()
df["temperature"] = df[[c for c in df.columns if c.startswith("temperature_station")]].mean(axis=1)
df["humidity"] = df["humidity"].where(df["humidity"] <= 100).fillna(df["humidity"].median())
WEATHER = sorted(df["weather_condition"].unique())
cats = np.stack([df["weather_condition"].map({w: i for i, w in enumerate(WEATHER)}).to_numpy(),
                 df["date"].dt.month.to_numpy() - 1, df["date"].dt.dayofweek.to_numpy()], 1)
CARD = [len(WEATHER), 12, 7]
num = df[["temperature", "humidity"]].to_numpy(np.float32)
target = df["electricity_demand"].to_numpy(np.float32)
split = int(0.8 * len(df))                                             # time-ordered: the last year validates
num = (num - num[:split].mean(0)) / num[:split].std(0)
t_mu, t_sd = target[:split].mean(), target[:split].std()
C, Xn, T = torch.from_numpy(cats).long(), torch.from_numpy(num), torch.from_numpy((target - t_mu) / t_sd)


class TabularNet(nn.Module):
    def __init__(self, cardinalities, n_numeric, embed_dim=None, hidden=64):
        super().__init__()
        self.embed = None if embed_dim is None else nn.ModuleList(nn.Embedding(c, embed_dim) for c in cardinalities)
        n_cat = sum(cardinalities) if embed_dim is None else embed_dim * len(cardinalities)
        self.mlp = nn.Sequential(nn.Linear(n_cat + n_numeric, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, cats, num):
        if self.embed is None:
            parts = [F.one_hot(cats[:, j], c).float() for j, c in enumerate(CARD)]
        else:
            parts = [emb(cats[:, j]) for j, emb in enumerate(self.embed)]
        return self.mlp(torch.cat(parts + [num], 1)).squeeze(1)


def train_tabular(model, name, epochs=60 if not QUICK else 10):
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    wandb.init(project=PROJECT, name=name, tags=["part7"], config={"params": sum(p.numel() for p in model.parameters())})
    try:
        for epoch in range(1, epochs + 1):
            model.train()
            for idx in torch.randperm(split).split(64):
                loss = F.mse_loss(model(C[idx], Xn[idx]), T[idx])
                opt.zero_grad(); loss.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                rmse = ((model(C[split:], Xn[split:]) - T[split:]) ** 2).mean().sqrt().item() * t_sd
            wandb.log({"epoch": epoch, "val/rmse": rmse})
        wandb.summary["val_rmse"] = rmse
    finally:
        wandb.finish()
    return rmse


onehot, embed = TabularNet(CARD, 2), TabularNet(CARD, 2, embed_dim=2)
print(f"one-hot parameters  : {sum(p.numel() for p in onehot.parameters())}")
print(f"embedding parameters: {sum(p.numel() for p in embed.parameters())}")
print(f"a 10,000-value column into 256 hidden units: one-hot {10_000 * 256:,} weights, "
      f"embedding d=16 {10_000 * 16 + 16 * 256:,}")
print(f"val RMSE  one-hot {train_tabular(onehot, 'p7-onehot'):.2f}   embedding {train_tabular(embed, 'p7-embedding'):.2f}"
      f"   (demand is in the hundreds; the constant-mean RMSE is {float(np.std(target[split:])):.2f})")

In [ ]:
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, emb, labels, color in ((axes[0], embed.embed[1], MONTHS, PALETTE[0]), (axes[1], embed.embed[2], DAYS, PALETTE[1])):
    w = emb.weight.detach().numpy()
    ax.scatter(w[:, 0], w[:, 1], color=color, s=40)
    for k, lab in enumerate(labels):
        ax.annotate(lab, w[k], textcoords="offset points", xytext=(5, 4), fontsize=10)
    ax.set_xlabel("embedding dimension 1"); ax.set_ylabel("embedding dimension 2")
axes[0].set_title("month"); axes[1].set_title("day of the week")
plt.tight_layout()
wandb.init(project=PROJECT, name="p7-embedding-plot", tags=["part7"])
wandb.log({"embeddings": wandb.Image(fig)})
wandb.finish()
plt.show()

**Question 7.** Do adjacent months sit next to each other in the learned 2-D
embedding, and which two groups of days does the weekday embedding separate? Does
that match the demand-by-month and demand-by-weekday bars of Lab 2?

*Your answer:*

---

## Part 8 — The 60-second budget

Challenge 8 calls `train(X_train, y_train)` with the 60,000 training images —
`X_train` uint8 `(60000, 28, 28)`, `y_train` int64 `(60000, 1)` — and then
`predict(X_test)` with the 10,000 test images, each under its **own 60 s deadline**,
on **3 CPU cores and 3 GiB** with no GPU. A deadline missed or an exception raised
scores 0. From here on the notebook uses the full task, built exactly as the
challenge builds it, and times what the challenge times.

Three candidates get the same training budget: a linear softmax regression, the
baseline MLP, and a wider MLP with BatchNorm. Each trains one epoch at a constant
learning rate to measure the machine, then anneals the learning rate to zero over
the epochs that still fit, and stops at the budget whatever happens. A fast machine
gets more epochs, a slow one fewer — the code adapts, the deadline does not.
`torch.set_num_threads(3)` is not cosmetic: in a container limited to 3 cores torch
would otherwise start one thread per core of the host and lose time fighting for them.

In [ ]:
TRAIN_BUDGET_S = 8.0 if QUICK else 40.0            # of the 60 s deadline; the grading pod is slower than a laptop
full = make_task(mnist_train.data.numpy(), mnist_train.targets.numpy(),
                 mnist_test.data.numpy(), mnist_test.targets.numpy(), seed=1)
print({k: (v.shape, v.dtype) for k, v in full.items()})


def budget_train(model, X, y, budget_s, lr=2e-3, batch=256, label_smoothing=0.05, log=None):
    """AdamW; epoch 1 at constant LR measures the speed, the rest anneal to 0 within the budget."""
    t0 = time.perf_counter()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = math.ceil(len(X) / batch)

    def epoch(sched=None):
        model.train()
        for idx in torch.randperm(len(X)).split(batch):
            loss = F.cross_entropy(model(X[idx]), y[idx], label_smoothing=label_smoothing)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            if sched is not None:
                sched.step()
            if time.perf_counter() - t0 > budget_s:
                return False
        return True

    t_epoch = time.perf_counter()
    epoch()
    per_epoch = time.perf_counter() - t_epoch
    n_more = max(0, int((budget_s - (time.perf_counter() - t0)) // per_epoch))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, n_more * steps))
    done = 1
    for _ in range(n_more):
        if not epoch(sched):
            break
        done += 1
        if log:
            log(done)
    model.eval()
    return done, per_epoch


X_full, y_full = to_tensors(full["X_train"], full["y_train"])
mu, sd = X_full.mean(), X_full.std()
X_full, X_test = (X_full - mu) / sd, (to_tensors(full["X_test"]) - mu) / sd
y_test = torch.from_numpy(full["y_test"].astype(np.int64))

CANDIDATES = {"linear-softmax": lambda: nn.Linear(784, 10),
              "mlp-256-256": lambda: build_model({**CONFIG, "hidden": (256, 256)}),
              "mlp-512-256-bn": lambda: build_model({**CONFIG, "hidden": (512, 256), "norm": "batchnorm"})}
budget_rows = {}
for name, make in CANDIDATES.items():
    torch.manual_seed(0)
    model = make()
    wandb.init(project=PROJECT, name=f"p8-{name}", tags=["part8"], config={"budget_s": TRAIN_BUDGET_S, "threads": torch.get_num_threads()})
    try:
        t0 = time.perf_counter()
        epochs_done, per_epoch = budget_train(model, X_full, y_full, TRAIN_BUDGET_S,
                                              log=lambda e: wandb.log({"epochs_done": e, "seconds": time.perf_counter() - t0}))
        train_s = time.perf_counter() - t0
        t0 = time.perf_counter()
        with torch.no_grad():
            acc = (torch.cat([model(xb).argmax(1) for xb in X_test.split(4096)]) == y_test).float().mean().item()
        predict_s = time.perf_counter() - t0
        budget_rows[name] = {"accuracy": acc, "train_s": train_s, "predict_s": predict_s, "epochs": epochs_done,
                             "s_per_epoch": per_epoch, "params": sum(p.numel() for p in model.parameters())}
        wandb.summary.update(budget_rows[name])
    finally:
        wandb.finish()
budget_table = pd.DataFrame(budget_rows).T
print(f"budget {TRAIN_BUDGET_S:.0f} s on {torch.get_num_threads()} threads")
budget_table.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
for (name, row), color in zip(budget_rows.items(), PALETTE):
    ax.scatter(row["train_s"], row["accuracy"], s=70, color=color, label=f"{name} ({int(row['epochs'])} epochs)")
ax.axvline(60, color="#c3c2b7", ls="--"); ax.text(60, ax.get_ylim()[0], " 60 s deadline", va="bottom", fontsize=9, color="#52514e")
ax.set_xlim(0, 65); ax.set_xlabel("train() seconds"); ax.set_ylabel("test accuracy"); ax.legend(loc="lower left"); plt.show()

**Question 8.** Which candidate reached the best accuracy inside the budget on your
machine, with how many epochs, and at how many seconds per epoch? Would the same
budget on a machine twice as slow change your choice?

*Your answer:*

---

## Part 9 — `agent.py`

The challenge imports `agent.py`, builds `Agent()` once, and calls `train` then
`predict`. The file below is the winning candidate of Part 8 as an agent. Five things
in it are rules, not choices:

- **The template's signatures.** The upload validator compares your methods with the
  challenge's template by method name *and parameter names*:
  `__init__(self, output_dim: int = 10, seed=None)`, `train(self, X_train, y_train)`,
  `predict(self, X_test)`. Rename `X_test` and the upload is rejected with
  "wrong signature".
- **A fresh model on every `train()` call.** If your accuracy reaches 0.98 the
  challenge calls `train` and `predict` again on a permuted Fashion-MNIST task and
  expects at least 0.40 there; a model kept from the previous call, or anything
  that assumes digits, scores −1 as a detected cheat.
- **Statistics from the arrays that arrive.** Mean and standard deviation are
  computed on this task's `X_train`, never hard-coded.
- **The wall-clock guard.** Training stops at `TRAIN_BUDGET_S` wherever it is.
- **`model.eval()` before predicting**, and `predict` returns a list of ints.

Edit the constants at the top of the file; the widths, the learning rate, the label
smoothing and the budget are the knobs Parts 1–8 were about. The harness in the
second cell is the challenge's `evaluate` loop: it times both calls against the 60 s
deadlines, computes the accuracy, and runs the Fashion-MNIST canary when the
accuracy reaches 0.98 — so a second `train()` on different data is part of the test.

In [ ]:
%%writefile agent.py
"""1 Minute Permuted MNIST (challenge 8) -- an MLP trained inside a wall-clock budget.

The challenge calls train(X_train, y_train) then predict(X_test), each under its own
60 s deadline, on 3 CPU cores and 3 GiB, no GPU. X is uint8 (N, 28, 28) with the
pixel positions permuted, y is int64 (N, 1) with the label meanings permuted.
Every call is a new task: the model, the statistics and the label set are all
rebuilt from the arrays that arrive.

The upload validator compares the methods with the challenge's template by name and
parameter names: keep __init__(self, output_dim: int = 10, seed=None),
train(self, X_train, y_train) and predict(self, X_test) spelled exactly like this.
"""
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

TRAIN_BUDGET_S = 40.0    # of the 60 s train() deadline: the grading pod is slower than a laptop
THREADS = 3              # agent_cpu_limit is 3000m; torch would otherwise count the host's cores
HIDDEN = (512, 256)
BATCH = 256
LR = 2e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05


class Agent:
    def __init__(self, output_dim: int = 10, seed=None):
        torch.set_num_threads(THREADS)
        self.seed = 0 if seed is None else seed
        self.model = None
        self.mu = None
        self.sd = None

    @staticmethod
    def _flatten(X):
        X = np.asarray(X)
        return torch.from_numpy(X.reshape(len(X), -1).astype(np.float32) / 255.0)

    def train(self, X_train, y_train):
        t0 = time.perf_counter()
        torch.manual_seed(self.seed)
        X = self._flatten(X_train)
        y = torch.from_numpy(np.asarray(y_train).reshape(-1).astype(np.int64))
        self.mu, self.sd = X.mean(), X.std()                # one scalar each, from THIS task's training set
        X = (X - self.mu) / self.sd
        n_classes = int(y.max()) + 1

        layers, d = [], X.shape[1]
        for h in HIDDEN:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU()]
            d = h
        layers.append(nn.Linear(d, n_classes))
        self.model = nn.Sequential(*layers)                   # a FRESH model on every call
        opt = torch.optim.AdamW(self.model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        n = len(X)
        steps = (n + BATCH - 1) // BATCH

        def epoch(sched=None):
            self.model.train()
            for idx in torch.randperm(n).split(BATCH):
                loss = F.cross_entropy(self.model(X[idx]), y[idx],
                                       label_smoothing=LABEL_SMOOTHING)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                if sched is not None:
                    sched.step()
                if time.perf_counter() - t0 > TRAIN_BUDGET_S:   # the wall-clock guard
                    return False
            return True

        # Epoch 1 at a constant LR measures this machine; the epochs that still fit
        # in the budget anneal the LR to zero, so training ends on a low LR wherever it stops.
        t_epoch = time.perf_counter()
        epoch()
        per_epoch = time.perf_counter() - t_epoch
        n_more = max(0, int((TRAIN_BUDGET_S - (time.perf_counter() - t0)) // per_epoch))
        if n_more > 0:
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_more * steps)
            for _ in range(n_more):
                if not epoch(sched):
                    break
        self.model.eval()                                      # BatchNorm uses its running statistics
        self.epochs_run = 1 + n_more

    def predict(self, X_test):
        X = (self._flatten(X_test) - self.mu) / self.sd
        with torch.no_grad():
            pred = torch.cat([self.model(xb).argmax(1) for xb in X.split(4096)])
        return pred.tolist()

In [ ]:
import importlib

import agent
importlib.reload(agent)                              # pick up any edit to agent.py
if QUICK:
    agent.TRAIN_BUDGET_S = 8.0


def evaluate_agent(AgentClass, task, deadline_s=60.0):
    """What the challenge does: time train() and predict() against the deadlines, then score."""
    bot = AgentClass()
    t0 = time.perf_counter(); bot.train(task["X_train"], task["y_train"]); train_s = time.perf_counter() - t0
    t0 = time.perf_counter(); pred = bot.predict(task["X_test"]); predict_s = time.perf_counter() - t0
    pred = np.asarray(pred).flatten()
    assert pred.shape == task["y_test"].shape and pred.dtype.kind in "iu", (pred.shape, pred.dtype)
    row = {"accuracy": float((pred == task["y_test"]).mean()), "train_s": train_s, "predict_s": predict_s,
           "within_deadlines": train_s < deadline_s and predict_s < deadline_s, "epochs": getattr(bot, "epochs_run", None)}
    if row["accuracy"] >= 0.98:                       # the challenge's cheat check: a different dataset, same agent
        fashion_tr = torchvision.datasets.FashionMNIST("data", train=True, download=True)
        fashion_te = torchvision.datasets.FashionMNIST("data", train=False, download=True)
        canary = make_task(fashion_tr.data.numpy(), fashion_tr.targets.numpy(),
                           fashion_te.data.numpy(), fashion_te.targets.numpy(), seed=2)
        bot.train(canary["X_train"], canary["y_train"])
        row["canary_accuracy"] = float((np.asarray(bot.predict(canary["X_test"])).flatten() == canary["y_test"]).mean())
        row["canary_pass"] = row["canary_accuracy"] >= 0.40
    return row


result = evaluate_agent(agent.Agent, full)
wandb.init(project=PROJECT, name="p9-agent", tags=["part9"], config={"threads": torch.get_num_threads(), "budget_s": agent.TRAIN_BUDGET_S})
wandb.summary.update(result); wandb.finish()
print(f"threads {torch.get_num_threads()}, budget {agent.TRAIN_BUDGET_S:.0f} s")
pd.Series(result).to_frame("agent.py")

**Question 9.** Report your agent's accuracy, `train_s` and `predict_s`, with the
thread count and the machine. What would the canary accuracy have been if `train()`
had kept the model of the first call and only continued training it — and why?

*Your answer:*

---

## Part 10 — Submit

The ML-Arena key comes from Colab's *Secrets* panel as `MLARENA_API_KEY` (ML-Arena,
Profile → API Keys; it starts with `mlk_user_`), or from your environment. The cell
does nothing if it is missing. `runtime={"language": "python", "framework": "torch"}`
picks the torch 2.12 runtime; `wait=True` returns once the evaluation has settled,
which takes a few minutes: the pod is scheduled, then your `train` and `predict` run
under the two deadlines, then the canary if you passed 0.98.

In [ ]:
MLARENA_API_KEY = os.environ.get("MLARENA_API_KEY") or colab_secret("MLARENA_API_KEY")
if not MLARENA_API_KEY:
    print("MLARENA_API_KEY is not set: add it to Colab Secrets (ML-Arena, Profile -> API Keys) and rerun this cell.")
else:
    import mlarena
    client = mlarena.connect(api_key=MLARENA_API_KEY)
    submission = client.submit(8, files=["agent.py"], submission_name="lab4-mlp",
                               runtime={"language": "python", "framework": "torch"}, wait=True)
    status = submission["status"]
    print(f"submission {submission['submission_id']}: {status['status']} -- {status['last_status_message']}")
    print(client.leaderboard(8, top=5))

---

## What goes in the deliverable

- Your copy of this notebook, run end to end with `QUICK = False`, with a
  one-sentence answer under each of the nine questions.
- The link to your W&B project (or the synced offline runs) — the grader opens the
  workspace and looks for the comparisons named above.
- `agent.py`, its Part 9 numbers **with the thread count and the machine**, and
  your submission on the leaderboard of challenge 8, above the `__benchmark__` row.
- `lab4_results.json`, written by the last cell: every summary, every per-epoch log,
  the range test, the budget table and the agent's numbers.

In [ ]:
import json

with open("lab4_results.json", "w") as f:
    json.dump({"quick": QUICK, "threads": torch.get_num_threads(), "results": RESULTS, "history": HISTORY,
               "lr_test": LR_TEST, "budget": budget_rows, "budget_s": TRAIN_BUDGET_S, "agent": result,
               "embeddings": {"month": embed.embed[1].weight.tolist(), "weekday": embed.embed[2].weight.tolist()}}, f, indent=1)
print("lab4_results.json", os.path.getsize("lab4_results.json"), "bytes")